In [1]:
print("hello world")

hello world


In [15]:
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
import numpy as np
from typing import List

# Pydantic Models
class FinancialMetrics(BaseModel):
    ticker: str
    report_period: str
    price_to_earnings_ratio: float | None
    price_to_book_ratio: float | None
    price_to_sales_ratio: float | None
    operating_margin: float | None
    net_margin: float | None
    return_on_equity: float | None
    revenue_growth: float | None
    earnings_growth: float | None
    book_value_growth: float | None
    current_ratio: float | None
    debt_to_equity: float | None
    free_cash_flow_per_share: float | None
    earnings_per_share: float | None

class FinancialMetricsResponse(BaseModel):
    financial_metrics: List[FinancialMetrics]

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        # Handle percentage values
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        # Handle regular numeric strings
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

# Main calculation function
def calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker='INFY'):
    # Clean the dataframes
    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    # Get all unique dates
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    
    # Assuming constant number of shares (in millions) - need actual data for accurate calculation
    shares_outstanding = pnl_df.loc['Net Profit\xa0+'] / pnl_df.loc['EPS in Rs'] * 1000000
    
    metrics_list = []
    
    for i, date in enumerate(dates):
        # Get values for current period from Balance Sheet
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        borrowings = bs_df.loc['Borrowings\xa0+'][date]
        other_liab = bs_df.loc['Other Liabilities\xa0+'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        
        # Get values from PNL Statement
        sales = pnl_df.loc['Sales\xa0+'][date]
        op_profit = pnl_df.loc['Operating Profit'][date]
        net_profit = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        # Get values from Cash Flow Statement
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        
        # Calculate book value
        book_value = equity + reserves
        
        # Calculate market cap (approximation using P/E ratio)
        market_cap = net_profit * 20  # Assuming average P/E of 20
        price = market_cap / shares_outstanding[date]
        
        # Calculate metrics
        pe_ratio = price / eps if eps != 0 else None
        pb_ratio = price / (book_value / shares_outstanding[date]) if book_value != 0 else None
        ps_ratio = price / (sales / shares_outstanding[date]) if sales != 0 else None
        
        op_margin = (op_profit / sales) * 100 if sales != 0 else None
        net_margin = (net_profit / sales) * 100 if sales != 0 else None
        roe = (net_profit / book_value) * 100 if book_value != 0 else None
        
        # Growth metrics (need previous period)
        rev_growth = None
        earn_growth = None
        bv_growth = None
        if i > 0:
            prev_date = dates[i-1]
            prev_sales = pnl_df.loc['Sales\xa0+'][prev_date]
            prev_profit = pnl_df.loc['Net Profit\xa0+'][prev_date]
            prev_bv = bs_df.loc['Equity Capital'][prev_date] + bs_df.loc['Reserves'][prev_date]
            
            rev_growth = ((sales - prev_sales) / prev_sales * 100) if prev_sales != 0 else None
            earn_growth = ((net_profit - prev_profit) / prev_profit * 100) if prev_profit != 0 else None
            bv_growth = ((book_value - prev_bv) / prev_bv * 100) if prev_bv != 0 else None
        
        # Current ratio (approximating current assets as Other Assets)
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liab = other_liab
        current_ratio = current_assets / current_liab if current_liab != 0 else None
        
        # Debt to equity
        de_ratio = borrowings / book_value if book_value != 0 else None
        
        # Free cash flow per share
        fcf = cash_op + cash_inv  # Simplified FCF calculation
        fcf_per_share = fcf / shares_outstanding[date] if shares_outstanding[date] != 0 else None
        
        # Create metrics object
        metrics = FinancialMetrics(
            ticker=ticker,
            report_period=date,
            price_to_earnings_ratio=round(pe_ratio, 2) if pe_ratio else None,
            price_to_book_ratio=round(pb_ratio, 2) if pb_ratio else None,
            price_to_sales_ratio=round(ps_ratio, 2) if ps_ratio else None,
            operating_margin=round(op_margin, 2) if op_margin else None,
            net_margin=round(net_margin, 2) if net_margin else None,
            return_on_equity=round(roe, 2) if roe else None,
            revenue_growth=round(rev_growth, 2) if rev_growth else None,
            earnings_growth=round(earn_growth, 2) if earn_growth else None,
            book_value_growth=round(bv_growth, 2) if bv_growth else None,
            current_ratio=round(current_ratio, 2) if current_ratio else None,
            debt_to_equity=round(de_ratio, 2) if de_ratio else None,
            free_cash_flow_per_share=round(fcf_per_share, 2) if fcf_per_share else None,
            earnings_per_share=round(eps, 2) if eps else None
        )
        metrics_list.append(metrics)
    
    return FinancialMetricsResponse(financial_metrics=metrics_list)

# Load data and calculate metrics
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

# Calculate metrics
result = calculate_financial_metrics(bs_df, cf_df, pnl_df)

# Display results
for metric in result.financial_metrics:
    print(f"\nPeriod: {metric.report_period}")
    print(metric.model_dump_json(indent=4))


Period: 2013-03-01
{
    "ticker": "INFY",
    "report_period": "2013-03-01",
    "price_to_earnings_ratio": 0.0,
    "price_to_book_ratio": 5.06,
    "price_to_sales_ratio": 4.96,
    "operating_margin": 29.97,
    "net_margin": 24.8,
    "return_on_equity": 25.28,
    "revenue_growth": null,
    "earnings_growth": null,
    "book_value_growth": null,
    "current_ratio": 4.78,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 0.0,
    "earnings_per_share": 19.84
}

Period: 2014-03-01
{
    "ticker": "INFY",
    "report_period": "2014-03-01",
    "price_to_earnings_ratio": 0.0,
    "price_to_book_ratio": 4.84,
    "price_to_sales_ratio": 4.6,
    "operating_margin": 28.27,
    "net_margin": 22.99,
    "return_on_equity": 24.22,
    "revenue_growth": 20.61,
    "earnings_growth": 11.83,
    "book_value_growth": 16.73,
    "current_ratio": 3.7,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 0.0,
    "earnings_per_share": 22.3
}

Period: 2015-03-01
{
    "ticker":

In [4]:
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

In [12]:
print("pnl_df index values:")
print(bs_df.index.tolist())

pnl_df index values:
['Equity Capital', 'Reserves', 'Borrowings\xa0+', 'Other Liabilities\xa0+', 'Total Liabilities', 'Fixed Assets\xa0+', 'CWIP', 'Investments', 'Other Assets\xa0+', 'Total Assets']


In [20]:
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
import numpy as np
from typing import List
import yfinance as yf
from datetime import datetime, timedelta

# Pydantic Models
class FinancialMetrics(BaseModel):
    ticker: str
    report_period: str
    price_to_earnings_ratio: float | None
    price_to_book_ratio: float | None
    price_to_sales_ratio: float | None
    operating_margin: float | None
    net_margin: float | None
    return_on_equity: float | None
    revenue_growth: float | None
    earnings_growth: float | None
    book_value_growth: float | None
    current_ratio: float | None
    debt_to_equity: float | None
    free_cash_flow_per_share: float | None
    earnings_per_share: float | None

class FinancialMetricsResponse(BaseModel):
    financial_metrics: List[FinancialMetrics]

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

# Fetch historical price from Yahoo Finance for a specific date
def get_closest_price(ticker, date_str):
    stock = yf.Ticker(ticker)
    date = pd.to_datetime(date_str).tz_localize('UTC')  # Make date timezone-aware (UTC)
    start_date = date - timedelta(days=5)  # Look back 5 days
    end_date = date + timedelta(days=5)    # Look forward 5 days
    hist = stock.history(start=start_date, end=end_date)
    if not hist.empty:
        # Find the index of the closest date
        idx = np.abs((hist.index - date).days).argmin()  # Use np.abs on the days array
        return hist['Close'].iloc[idx]  # Return the Close price for that date
    return None

# Main calculation function
def calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker='INFY'):
    # Clean the dataframes
    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    # Get all unique dates (reporting periods)
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    
    # Get shares outstanding from Yahoo Finance (latest value)
    stock = yf.Ticker(ticker)
    shares_outstanding = stock.info['sharesOutstanding'] / 1000000  # Convert to millions
    
    metrics_list = []
    
    for i, date in enumerate(dates):
        # Get values for current period from Balance Sheet
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        borrowings = bs_df.loc['Borrowings\xa0+'][date]
        other_liab = bs_df.loc['Other Liabilities\xa0+'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        
        # Get values from PNL Statement
        sales = pnl_df.loc['Sales\xa0+'][date]
        op_profit = pnl_df.loc['Operating Profit'][date]
        net_profit = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        # Get values from Cash Flow Statement
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        
        # Calculate book value
        book_value = equity + reserves
        
        # Get actual market price from Yahoo Finance
        price = get_closest_price(ticker, date)
        if price is not None:
            market_cap = price * shares_outstanding  # Actual market cap
        else:
            market_cap = net_profit * 20  # Fallback approximation
            price = market_cap / shares_outstanding
        
        # Calculate metrics
        pe_ratio = price / eps if eps != 0 else None
        pb_ratio = price / (book_value / shares_outstanding) if book_value != 0 else None
        ps_ratio = price / (sales / shares_outstanding) if sales != 0 else None
        
        op_margin = (op_profit / sales) * 100 if sales != 0 else None
        net_margin = (net_profit / sales) * 100 if sales != 0 else None
        roe = (net_profit / book_value) * 100 if book_value != 0 else None
        
        # Growth metrics (need previous period)
        rev_growth = None
        earn_growth = None
        bv_growth = None
        if i > 0:
            prev_date = dates[i-1]
            prev_sales = pnl_df.loc['Sales\xa0+'][prev_date]
            prev_profit = pnl_df.loc['Net Profit\xa0+'][prev_date]
            prev_bv = bs_df.loc['Equity Capital'][prev_date] + bs_df.loc['Reserves'][prev_date]
            prev_price = get_closest_price(ticker, prev_date)
            if prev_price is not None:
                market_cap_prev = prev_price * shares_outstanding
            else:
                market_cap_prev = prev_profit * 20  # Fallback approximation
                prev_price = market_cap_prev / shares_outstanding
            
            rev_growth = ((sales - prev_sales) / prev_sales * 100) if prev_sales != 0 else None
            earn_growth = ((net_profit - prev_profit) / prev_profit * 100) if prev_profit != 0 else None
            bv_growth = ((book_value - prev_bv) / prev_bv * 100) if prev_bv != 0 else None
        
        # Current ratio (approximating current assets as Other Assets)
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liab = other_liab
        current_ratio = current_assets / current_liab if current_liab != 0 else None
        
        # Debt to equity
        de_ratio = borrowings / book_value if book_value != 0 else None
        
        # Free cash flow per share
        fcf = cash_op + cash_inv  # Simplified FCF calculation
        fcf_per_share = fcf / shares_outstanding if shares_outstanding != 0 else None
        
        # Create metrics object
        metrics = FinancialMetrics(
            ticker=ticker,
            report_period=date,
            price_to_earnings_ratio=round(pe_ratio, 2) if pe_ratio else None,
            price_to_book_ratio=round(pb_ratio, 2) if pb_ratio else None,
            price_to_sales_ratio=round(ps_ratio, 2) if ps_ratio else None,
            operating_margin=round(op_margin, 2) if op_margin else None,
            net_margin=round(net_margin, 2) if net_margin else None,
            return_on_equity=round(roe, 2) if roe else None,
            revenue_growth=round(rev_growth, 2) if rev_growth else None,
            earnings_growth=round(earn_growth, 2) if earn_growth else None,
            book_value_growth=round(bv_growth, 2) if bv_growth else None,
            current_ratio=round(current_ratio, 2) if current_ratio else None,
            debt_to_equity=round(de_ratio, 2) if de_ratio else None,
            free_cash_flow_per_share=round(fcf_per_share, 2) if fcf_per_share else None,
            earnings_per_share=round(eps, 2) if eps else None
        )
        metrics_list.append(metrics)
    
    return FinancialMetricsResponse(financial_metrics=metrics_list)

# Load data and calculate metrics
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

# Calculate metrics
result = calculate_financial_metrics(bs_df, cf_df, pnl_df)

# Display results
for metric in result.financial_metrics:
    print(f"\nPeriod: {metric.report_period}")
    print(metric.model_dump_json(indent=4))


Period: 2013-03-01
{
    "ticker": "INFY",
    "report_period": "2013-03-01",
    "price_to_earnings_ratio": 0.25,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.56,
    "operating_margin": 29.97,
    "net_margin": 24.8,
    "return_on_equity": 25.28,
    "revenue_growth": null,
    "earnings_growth": null,
    "book_value_growth": null,
    "current_ratio": 4.78,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 1.0,
    "earnings_per_share": 19.84
}

Period: 2014-03-01
{
    "ticker": "INFY",
    "report_period": "2014-03-01",
    "price_to_earnings_ratio": 0.26,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.54,
    "operating_margin": 28.27,
    "net_margin": 22.99,
    "return_on_equity": 24.22,
    "revenue_growth": 20.61,
    "earnings_growth": 11.83,
    "book_value_growth": 16.73,
    "current_ratio": 3.7,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 1.66,
    "earnings_per_share": 22.3
}

Period: 2015-03-01
{
    "tick

In [21]:
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
import numpy as np
from typing import List
import yfinance as yf
from datetime import datetime, timedelta

# Pydantic Models
class FinancialMetrics(BaseModel):
    ticker: str
    report_period: str
    price_to_earnings_ratio: float | None
    price_to_book_ratio: float | None
    price_to_sales_ratio: float | None
    operating_margin: float | None
    net_margin: float | None
    return_on_equity: float | None
    revenue_growth: float | None
    earnings_growth: float | None
    book_value_growth: float | None
    current_ratio: float | None
    debt_to_equity: float | None
    free_cash_flow_per_share: float | None
    earnings_per_share: float | None

class FinancialMetricsResponse(BaseModel):
    financial_metrics: List[FinancialMetrics]

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

# Fetch historical price from Yahoo Finance for a specific date
def get_closest_price(ticker, date_str):
    stock = yf.Ticker(ticker)
    date = pd.to_datetime(date_str).tz_localize('UTC')  # Make date timezone-aware (UTC)
    start_date = date - timedelta(days=5)  # Look back 5 days
    end_date = date + timedelta(days=5)    # Look forward 5 days
    hist = stock.history(start=start_date, end=end_date)
    if not hist.empty:
        idx = np.abs((hist.index - date).days).argmin()
        return hist['Close'].iloc[idx]
    return None

# Main calculation function
def calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker='INFY'):
    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    stock = yf.Ticker(ticker)
    shares_outstanding = stock.info['sharesOutstanding'] / 1000000
    
    metrics_list = []
    
    for i, date in enumerate(dates):
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        borrowings = bs_df.loc['Borrowings\xa0+'][date]
        other_liab = bs_df.loc['Other Liabilities\xa0+'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        
        sales = pnl_df.loc['Sales\xa0+'][date]
        op_profit = pnl_df.loc['Operating Profit'][date]
        net_profit = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        
        book_value = equity + reserves
        price = get_closest_price(ticker, date)
        if price is not None:
            market_cap = price * shares_outstanding
        else:
            market_cap = net_profit * 20
            price = market_cap / shares_outstanding
        
        pe_ratio = price / eps if eps != 0 else None
        pb_ratio = price / (book_value / shares_outstanding) if book_value != 0 else None
        ps_ratio = price / (sales / shares_outstanding) if sales != 0 else None
        
        op_margin = (op_profit / sales) * 100 if sales != 0 else None
        net_margin = (net_profit / sales) * 100 if sales != 0 else None
        roe = (net_profit / book_value) * 100 if book_value != 0 else None
        
        rev_growth = None
        earn_growth = None
        bv_growth = None
        if i > 0:
            prev_date = dates[i-1]
            prev_sales = pnl_df.loc['Sales\xa0+'][prev_date]
            prev_profit = pnl_df.loc['Net Profit\xa0+'][prev_date]
            prev_bv = bs_df.loc['Equity Capital'][prev_date] + bs_df.loc['Reserves'][prev_date]
            prev_price = get_closest_price(ticker, prev_date)
            if prev_price is not None:
                market_cap_prev = prev_price * shares_outstanding
            else:
                market_cap_prev = prev_profit * 20
                prev_price = market_cap_prev / shares_outstanding
            
            rev_growth = ((sales - prev_sales) / prev_sales * 100) if prev_sales != 0 else None
            earn_growth = ((net_profit - prev_profit) / prev_profit * 100) if prev_profit != 0 else None
            bv_growth = ((book_value - prev_bv) / prev_bv * 100) if prev_bv != 0 else None
        
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liab = other_liab
        current_ratio = current_assets / current_liab if current_liab != 0 else None
        
        de_ratio = borrowings / book_value if book_value != 0 else None
        
        fcf = cash_op + cash_inv
        fcf_per_share = fcf / shares_outstanding if shares_outstanding != 0 else None
        
        metrics = FinancialMetrics(
            ticker=ticker,
            report_period=date,
            price_to_earnings_ratio=round(pe_ratio, 2) if pe_ratio else None,
            price_to_book_ratio=round(pb_ratio, 2) if pb_ratio else None,
            price_to_sales_ratio=round(ps_ratio, 2) if ps_ratio else None,
            operating_margin=round(op_margin, 2) if op_margin else None,
            net_margin=round(net_margin, 2) if net_margin else None,
            return_on_equity=round(roe, 2) if roe else None,
            revenue_growth=round(rev_growth, 2) if rev_growth else None,
            earnings_growth=round(earn_growth, 2) if earn_growth else None,
            book_value_growth=round(bv_growth, 2) if bv_growth else None,
            current_ratio=round(current_ratio, 2) if current_ratio else None,
            debt_to_equity=round(de_ratio, 2) if de_ratio else None,
            free_cash_flow_per_share=round(fcf_per_share, 2) if fcf_per_share else None,
            earnings_per_share=round(eps, 2) if eps else None
        )
        metrics_list.append(metrics)
    
    return FinancialMetricsResponse(financial_metrics=metrics_list)

# Function to evaluate metrics against thresholds
def evaluate_metrics(metrics):
    profitability_thresholds = [
        ('return_on_equity', 0.15),  # Strong ROE above 15%
        ('net_margin', 0.20),        # Healthy profit margins
        ('operating_margin', 0.15),  # Strong operating efficiency
    ]
    
    growth_thresholds = [
        ('revenue_growth', 0.10),    # 10% revenue growth
        ('earnings_growth', 0.10),   # 10% earnings growth
        ('book_value_growth', 0.10), # 10% book value growth
    ]
    
    valuation_thresholds = [
        ('price_to_earnings_ratio', 25), # Reasonable P/E ratio
        ('price_to_book_ratio', 3),      # Reasonable P/B ratio
        ('price_to_sales_ratio', 5),     # Reasonable P/S ratio
    ]
    
    # Convert metrics to a dictionary for easier access
    metrics_dict = metrics.dict()
    
    # Profitability evaluation
    profitability_score = 0
    for metric_name, threshold in profitability_thresholds:
        value = metrics_dict[metric_name]
        if value is not None and value > threshold:
            profitability_score += 1
    profitability_result = f"Profitability: {profitability_score}/{len(profitability_thresholds)} thresholds met"
    
    # Growth evaluation
    growth_score = 0
    for metric_name, threshold in growth_thresholds:
        value = metrics_dict[metric_name]
        if value is not None and value > threshold:
            growth_score += 1
    growth_result = f"Growth: {growth_score}/{len(growth_thresholds)} thresholds met"
    
    # Valuation evaluation
    valuation_score = 0
    for metric_name, threshold in valuation_thresholds:
        value = metrics_dict[metric_name]
        if value is not None and value < threshold:  # Note: < for valuation (lower is better)
            valuation_score += 1
    valuation_result = f"Valuation: {valuation_score}/{len(valuation_thresholds)} thresholds met"
    
    # Health score evaluation
    health_score = 0
    if metrics_dict['current_ratio'] and metrics_dict['current_ratio'] > 1.5:
        health_score += 1
    if metrics_dict['debt_to_equity'] and metrics_dict['debt_to_equity'] < 0.5:
        health_score += 1
    if (metrics_dict['free_cash_flow_per_share'] and metrics_dict['earnings_per_share'] and 
        metrics_dict['free_cash_flow_per_share'] > metrics_dict['earnings_per_share'] * 0.8):
        health_score += 1
    health_result = f"Health Score: {health_score}/3"
    
    return {
        "profitability": profitability_result,
        "growth": growth_result,
        "valuation": valuation_result,
        "health": health_result
    }

# Load data and calculate metrics
bs_df = pd.read_csv('financial_data/INFY_BS.csv', index_col=0)
cf_df = pd.read_csv('financial_data/INFY_CF.csv', index_col=0)
pnl_df = pd.read_csv('financial_data/INFY_PNL.csv', index_col=0)

# Calculate metrics
result = calculate_financial_metrics(bs_df, cf_df, pnl_df)

# Display results with threshold evaluation
for metric in result.financial_metrics:
    print(f"\nPeriod: {metric.report_period}")
    print(metric.model_dump_json(indent=4))
    evaluation = evaluate_metrics(metric)
    print("\nEvaluation:")
    for category, result in evaluation.items():
        print(f"  {result}")


Period: 2013-03-01
{
    "ticker": "INFY",
    "report_period": "2013-03-01",
    "price_to_earnings_ratio": 0.25,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.56,
    "operating_margin": 29.97,
    "net_margin": 24.8,
    "return_on_equity": 25.28,
    "revenue_growth": null,
    "earnings_growth": null,
    "book_value_growth": null,
    "current_ratio": 4.78,
    "debt_to_equity": null,
    "free_cash_flow_per_share": 1.0,
    "earnings_per_share": 19.84
}

Evaluation:
  Profitability: 3/3 thresholds met
  Growth: 0/3 thresholds met
  Valuation: 3/3 thresholds met
  Health Score: 1/3

Period: 2014-03-01
{
    "ticker": "INFY",
    "report_period": "2014-03-01",
    "price_to_earnings_ratio": 0.26,
    "price_to_book_ratio": 0.57,
    "price_to_sales_ratio": 0.54,
    "operating_margin": 28.27,
    "net_margin": 22.99,
    "return_on_equity": 24.22,
    "revenue_growth": 20.61,
    "earnings_growth": 11.83,
    "book_value_growth": 16.73,
    "current_ratio": 3.7,


/var/folders/2z/yqwddyc959v2vj85vf4bgzth0000gn/T/ipykernel_88501/2693647063.py:163: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.10/migration/
  metrics_dict = metrics.dict()


In [26]:
from langchain_core.messages import HumanMessage
from graph.state import AgentState, show_agent_reasoning
from utils.progress import progress
import json
import pandas as pd
import matplotlib.pyplot as plt
from pydantic import BaseModel
import numpy as np
from typing import List
import yfinance as yf
from datetime import datetime, timedelta
import os
import requests

# Pydantic Models
class FinancialMetrics(BaseModel):
    ticker: str
    report_period: str
    price_to_earnings_ratio: float | None
    price_to_book_ratio: float | None
    price_to_sales_ratio: float | None
    operating_margin: float | None
    net_margin: float | None
    return_on_equity: float | None
    revenue_growth: float | None
    earnings_growth: float | None
    book_value_growth: float | None
    current_ratio: float | None
    debt_to_equity: float | None
    free_cash_flow_per_share: float | None
    earnings_per_share: float | None

class FinancialMetricsResponse(BaseModel):
    financial_metrics: List[FinancialMetrics]

# Helper function to clean numeric values
def clean_numeric(value):
    if isinstance(value, str):
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

# Fetch historical price from Yahoo Finance for a specific date
def get_closest_price(ticker, date_str):
    stock = yf.Ticker(ticker)
    date = pd.to_datetime(date_str).tz_localize('UTC')
    start_date = date - timedelta(days=5)
    end_date = date + timedelta(days=5)
    hist = stock.history(start=start_date, end=end_date)
    if not hist.empty:
        idx = np.abs((hist.index - date).days).argmin()
        return hist['Close'].iloc[idx]
    return None

# Calculate financial metrics from CSV data
def calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker='INFY'):
    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    stock = yf.Ticker(ticker + '.NS')  # Append .NS for Indian stocks
    
    # Try to get shares outstanding from Yahoo Finance, fallback to estimation
    try:
        shares_outstanding = stock.info['sharesOutstanding'] / 1000000  # Convert to millions
    except KeyError:
        # Estimate shares outstanding using Equity Capital (in crores INR) and a price
        latest_date = dates[-1]  # Use the most recent date
        equity_capital = bs_df.loc['Equity Capital'][latest_date] * 10  # Convert crores to millions (1 crore = 10M)
        price = get_closest_price(ticker + '.NS', latest_date)
        if price and price > 0:
            shares_outstanding = equity_capital / price  # Estimate shares (millions)
        else:
            print(f"Warning: Could not estimate shares outstanding for {ticker}, using default 1000M")
            shares_outstanding = 1000  # Default fallback if no price
    
    metrics_list = []
    
    for i, date in enumerate(dates):
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        borrowings = bs_df.loc['Borrowings\xa0+'][date]
        other_liab = bs_df.loc['Other Liabilities\xa0+'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        
        sales = pnl_df.loc['Sales\xa0+'][date]
        op_profit = pnl_df.loc['Operating Profit'][date]
        net_profit = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        
        book_value = equity + reserves
        price = get_closest_price(ticker + '.NS', date)
        if price is not None:
            market_cap = price * shares_outstanding
        else:
            market_cap = net_profit * 20
            price = market_cap / shares_outstanding
        
        pe_ratio = price / eps if eps != 0 else None
        pb_ratio = price / (book_value / shares_outstanding) if book_value != 0 else None
        ps_ratio = price / (sales / shares_outstanding) if sales != 0 else None
        
        op_margin = (op_profit / sales) * 100 if sales != 0 else None
        net_margin = (net_profit / sales) * 100 if sales != 0 else None
        roe = (net_profit / book_value) * 100 if book_value != 0 else None
        
        rev_growth = None
        earn_growth = None
        bv_growth = None
        if i > 0:
            prev_date = dates[i-1]
            prev_sales = pnl_df.loc['Sales\xa0+'][prev_date]
            prev_profit = pnl_df.loc['Net Profit\xa0+'][prev_date]
            prev_bv = bs_df.loc['Equity Capital'][prev_date] + bs_df.loc['Reserves'][prev_date]
            prev_price = get_closest_price(ticker + '.NS', prev_date)
            if prev_price is not None:
                market_cap_prev = prev_price * shares_outstanding
            else:
                market_cap_prev = prev_profit * 20
                prev_price = market_cap_prev / shares_outstanding
            
            rev_growth = ((sales - prev_sales) / prev_sales * 100) if prev_sales != 0 else None
            earn_growth = ((net_profit - prev_profit) / prev_profit * 100) if prev_profit != 0 else None
            bv_growth = ((book_value - prev_bv) / prev_bv * 100) if prev_bv != 0 else None
        
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liab = other_liab
        current_ratio = current_assets / current_liab if current_liab != 0 else None
        
        de_ratio = borrowings / book_value if book_value != 0 else None
        
        fcf = cash_op + cash_inv
        fcf_per_share = fcf / shares_outstanding if shares_outstanding != 0 else None
        
        metrics = FinancialMetrics(
            ticker=ticker,
            report_period=date,
            price_to_earnings_ratio=round(pe_ratio, 2) if pe_ratio else None,
            price_to_book_ratio=round(pb_ratio, 2) if pb_ratio else None,
            price_to_sales_ratio=round(ps_ratio, 2) if ps_ratio else None,
            operating_margin=round(op_margin, 2) if op_margin else None,
            net_margin=round(net_margin, 2) if net_margin else None,
            return_on_equity=round(roe, 2) if roe else None,
            revenue_growth=round(rev_growth, 2) if rev_growth else None,
            earnings_growth=round(earn_growth, 2) if earn_growth else None,
            book_value_growth=round(bv_growth, 2) if bv_growth else None,
            current_ratio=round(current_ratio, 2) if current_ratio else None,
            debt_to_equity=round(de_ratio, 2) if de_ratio else None,
            free_cash_flow_per_share=round(fcf_per_share, 2) if fcf_per_share else None,
            earnings_per_share=round(eps, 2) if eps else None
        )
        metrics_list.append(metrics)
    
    return FinancialMetricsResponse(financial_metrics=metrics_list)

# Modified get_financial_metrics function for Indian stocks
def get_financial_metrics(
    ticker: str,
    end_date: str,
    period: str = "ttm",
    limit: int = 10,
) -> List[FinancialMetrics]:
    """Fetch financial metrics for Indian stocks using local CSV data and Yahoo Finance."""
    # Load CSV data from src/financial_data/
    try:
        bs_df = pd.read_csv(f'src/tools/financial_data/{ticker}_BS.csv', index_col=0)
        cf_df = pd.read_csv(f'src/tools/financial_data/{ticker}_CF.csv', index_col=0)
        pnl_df = pd.read_csv(f'src/tools/financial_data/{ticker}_PNL.csv', index_col=0)
    except FileNotFoundError as e:
        print(f"Error: CSV files for {ticker} not found in src/financial_data/ - {e}")
        return []

    # Calculate metrics
    metrics_response = calculate_financial_metrics(bs_df, cf_df, pnl_df, ticker)
    financial_metrics = metrics_response.financial_metrics

    if not financial_metrics:
        return []

    # Filter by end_date and limit
    end_date_dt = pd.to_datetime(end_date)
    filtered_metrics = [m for m in financial_metrics if pd.to_datetime(m.report_period) <= end_date_dt]
    filtered_metrics.sort(key=lambda x: x.report_period, reverse=True)
    
    # Handle period (currently only annual data, TTM not directly supported)
    if period.lower() == "ttm":
        return filtered_metrics[:min(limit, len(filtered_metrics))]
    else:
        return filtered_metrics[:min(limit, len(filtered_metrics))]

##### Fundamental Agent #####
def fundamentals_agent(state: AgentState):
    """Analyzes fundamental data and generates trading signals for multiple tickers."""
    data = state["data"]
    end_date = data["end_date"]
    tickers = data["tickers"]

    fundamental_analysis = {}

    for ticker in tickers:
        progress.update_status("fundamentals_agent", ticker, "Fetching financial metrics")

        financial_metrics = get_financial_metrics(
            ticker=ticker,
            end_date=end_date,
            period="ttm",
            limit=10,
        )

        if not financial_metrics:
            progress.update_status("fundamentals_agent", ticker, "Failed: No financial metrics found")
            continue

        metrics = financial_metrics[0]  # Most recent period

        signals = []
        reasoning = {}

        progress.update_status("fundamentals_agent", ticker, "Analyzing profitability")
        return_on_equity = metrics.return_on_equity
        net_margin = metrics.net_margin
        operating_margin = metrics.operating_margin

        thresholds = [
            (return_on_equity, 0.15),
            (net_margin, 0.20),
            (operating_margin, 0.15),
        ]
        profitability_score = sum(metric is not None and metric > threshold for metric, threshold in thresholds)

        signals.append("bullish" if profitability_score >= 2 else "bearish" if profitability_score == 0 else "neutral")
        reasoning["profitability_signal"] = {
            "signal": signals[0],
            "details": (f"ROE: {return_on_equity:.2%}" if return_on_equity else "ROE: N/A") + ", " + 
                       (f"Net Margin: {net_margin:.2%}" if net_margin else "Net Margin: N/A") + ", " + 
                       (f"Op Margin: {operating_margin:.2%}" if operating_margin else "Op Margin: N/A"),
        }

        progress.update_status("fundamentals_agent", ticker, "Analyzing growth")
        revenue_growth = metrics.revenue_growth
        earnings_growth = metrics.earnings_growth
        book_value_growth = metrics.book_value_growth

        thresholds = [
            (revenue_growth, 0.10),
            (earnings_growth, 0.10),
            (book_value_growth, 0.10),
        ]
        growth_score = sum(metric is not None and metric > threshold for metric, threshold in thresholds)

        signals.append("bullish" if growth_score >= 2 else "bearish" if growth_score == 0 else "neutral")
        reasoning["growth_signal"] = {
            "signal": signals[1],
            "details": (f"Revenue Growth: {revenue_growth:.2%}" if revenue_growth else "Revenue Growth: N/A") + ", " + 
                       (f"Earnings Growth: {earnings_growth:.2%}" if earnings_growth else "Earnings Growth: N/A"),
        }

        progress.update_status("fundamentals_agent", ticker, "Analyzing financial health")
        current_ratio = metrics.current_ratio
        debt_to_equity = metrics.debt_to_equity
        free_cash_flow_per_share = metrics.free_cash_flow_per_share
        earnings_per_share = metrics.earnings_per_share

        health_score = 0
        if current_ratio and current_ratio > 1.5:
            health_score += 1
        if debt_to_equity and debt_to_equity < 0.5:
            health_score += 1
        if free_cash_flow_per_share and earnings_per_share and free_cash_flow_per_share > earnings_per_share * 0.8:
            health_score += 1

        signals.append("bullish" if health_score >= 2 else "bearish" if health_score == 0 else "neutral")
        reasoning["financial_health_signal"] = {
            "signal": signals[2],
            "details": (f"Current Ratio: {current_ratio:.2f}" if current_ratio else "Current Ratio: N/A") + ", " + 
                       (f"D/E: {debt_to_equity:.2f}" if debt_to_equity else "D/E: N/A"),
        }

        progress.update_status("fundamentals_agent", ticker, "Analyzing valuation ratios")
        pe_ratio = metrics.price_to_earnings_ratio
        pb_ratio = metrics.price_to_book_ratio
        ps_ratio = metrics.price_to_sales_ratio

        thresholds = [
            (pe_ratio, 25),
            (pb_ratio, 3),
            (ps_ratio, 5),
        ]
        price_ratio_score = sum(metric is not None and metric > threshold for metric, threshold in thresholds)

        signals.append("bullish" if price_ratio_score >= 2 else "bearish" if price_ratio_score == 0 else "neutral")
        reasoning["price_ratios_signal"] = {
            "signal": signals[3],
            "details": (f"P/E: {pe_ratio:.2f}" if pe_ratio else "P/E: N/A") + ", " + 
                       (f"P/B: {pb_ratio:.2f}" if pb_ratio else "P/B: N/A") + ", " + 
                       (f"P/S: {ps_ratio:.2f}" if ps_ratio else "P/S: N/A"),
        }

        progress.update_status("fundamentals_agent", ticker, "Calculating final signal")
        bullish_signals = signals.count("bullish")
        bearish_signals = signals.count("bearish")

        if bullish_signals > bearish_signals:
            overall_signal = "bullish"
        elif bearish_signals > bullish_signals:
            overall_signal = "bearish"
        else:
            overall_signal = "neutral"

        confidence = round(max(bullish_signals, bearish_signals) / len(signals), 2) * 100

        fundamental_analysis[ticker] = {
            "signal": overall_signal,
            "confidence": confidence,
            "reasoning": reasoning,
        }

        progress.update_status("fundamentals_agent", ticker, "Done")

    message = HumanMessage(
        content=json.dumps(fundamental_analysis),
        name="fundamentals_agent",
    )

    if state["metadata"]["show_reasoning"]:
        show_agent_reasoning(fundamental_analysis, "Fundamental Analysis Agent")

    state["data"]["analyst_signals"]["fundamentals_agent"] = fundamental_analysis

    return {
        "messages": [message],
        "data": state["data"],
    }

# Example usage (for testing)
if __name__ == "__main__":
    state = {
        "data": {
            "end_date": "2023-03-01",
            "tickers": ["RELIANCE"],
            "analyst_signals": {}
        },
        "metadata": {"show_reasoning": True}
    }
    result = fundamentals_agent(state)
    print(result["messages"][0].content)

Error: CSV files for RELIANCE not found in src/financial_data/ - [Errno 2] No such file or directory: 'src/tools/financial_data/RELIANCE_BS.csv'

==========  Fundamental Analysis Agent  ==========
{}
{}


In [31]:
# Updated Jupyter Notebook Cell for Testing Ben Graham Agent

# Import required libraries
import pandas as pd
import yfinance as yf
from datetime import timedelta
import numpy as np
from pydantic import BaseModel
from typing import Optional, List
import json
import math

# Pydantic Models
class BenGrahamMetrics(BaseModel):
    ticker: str
    report_period: str
    earnings_per_share: Optional[float] = None
    revenue: Optional[float] = None
    net_income: Optional[float] = None
    book_value_per_share: Optional[float] = None
    total_assets: Optional[float] = None
    total_liabilities: Optional[float] = None
    current_assets: Optional[float] = None
    current_liabilities: Optional[float] = None
    dividends_per_share: Optional[float] = None
    outstanding_shares: Optional[float] = None

class BenGrahamMetricsResponse(BaseModel):
    metrics: list[BenGrahamMetrics]

# Utility Functions
def clean_numeric(value):
    if isinstance(value, str):
        if '%' in value:
            return float(value.replace('%', '').replace('"', '')) / 100
        return float(value.replace(',', '').replace('"', ''))
    return float(value) if value is not None else None

def get_closest_price(ticker, date_str):
    stock = yf.Ticker(ticker)
    date = pd.to_datetime(date_str).tz_localize('UTC')
    start_date = date - timedelta(days=5)
    end_date = date + timedelta(days=5)
    hist = stock.history(start=start_date, end=end_date)
    if not hist.empty:
        idx = np.abs((hist.index - date).days).argmin()
        return hist['Close'].iloc[idx]
    return None

# Ben Graham Metrics Function
def ben_graham_metrics(ticker: str, end_date: str, period: str = "annual", limit: int = 10) -> List[BenGrahamMetrics]:
    try:
        bs_df = pd.read_csv(f'tools/financial_data/{ticker}_BS.csv', index_col=0)
        cf_df = pd.read_csv(f'tools/financial_data/{ticker}_CF.csv', index_col=0)
        pnl_df = pd.read_csv(f'tools/financial_data/{ticker}_PNL.csv', index_col=0)
    except FileNotFoundError as e:
        print(f"Error: CSV files for {ticker} not found - {e}")
        return []
    
    if not ticker.endswith('.NS'):
        ticker = f"{ticker}.NS"

    for df in [bs_df, cf_df, pnl_df]:
        for col in df.columns:
            df[col] = df[col].apply(clean_numeric)
    
    dates = sorted(set(bs_df.columns) & set(cf_df.columns) & set(pnl_df.columns))
    stock = yf.Ticker(ticker)
    shares_outstanding = stock.info.get('sharesOutstanding', 0) / 1000000
    
    if shares_outstanding == 0:
        print(f"Warning: No shares outstanding data for {ticker}")
        return []

    metrics_list = []
    
    for date in dates:
        equity = bs_df.loc['Equity Capital'][date]
        reserves = bs_df.loc['Reserves'][date]
        total_assets = bs_df.loc['Total Assets'][date]
        total_liabilities = bs_df.loc['Borrowings\xa0+'][date] + bs_df.loc['Other Liabilities\xa0+'][date]
        current_assets = bs_df.loc['Other Assets\xa0+'][date]
        current_liabilities = bs_df.loc['Other Liabilities\xa0+'][date]
        
        revenue = pnl_df.loc['Sales\xa0+'][date]
        net_income = pnl_df.loc['Net Profit\xa0+'][date]
        eps = pnl_df.loc['EPS in Rs'][date]
        
        cash_op = cf_df.loc['Cash from Operating Activity\xa0+'][date]
        cash_inv = cf_df.loc['Cash from Investing Activity\xa0+'][date]
        cash_fin = cf_df.loc['Cash from Financing Activity\xa0+'][date]
        dividends_per_share = None
        if cash_fin < 0:
            dividends_per_share = abs(cash_fin) / shares_outstanding

        book_value = equity + reserves
        book_value_per_share = book_value / shares_outstanding if shares_outstanding != 0 else None

        metrics = BenGrahamMetrics(
            ticker=ticker,
            report_period=date,
            earnings_per_share=eps,
            revenue=revenue,
            net_income=net_income,
            book_value_per_share=book_value_per_share,
            total_assets=total_assets,
            total_liabilities=total_liabilities,
            current_assets=current_assets,
            current_liabilities=current_liabilities,
            dividends_per_share=dividends_per_share,
            outstanding_shares=shares_outstanding
        )
        metrics_list.append(metrics)
    
    end_date_dt = pd.to_datetime(end_date)
    filtered_metrics = [m for m in metrics_list if pd.to_datetime(m.report_period) <= end_date_dt]
    filtered_metrics.sort(key=lambda x: x.report_period, reverse=True)
    
    return filtered_metrics[:min(limit, len(filtered_metrics))]

# Simplified Ben Graham Agent for Testing
def test_ben_graham_agent(ticker: str, end_date: str):
    # Mock AgentState
    state = {
        "data": {"tickers": [ticker], "end_date": end_date, "analyst_signals": {}},
        "metadata": {"show_reasoning": True, "model_name": "mock", "model_provider": "mock"}
    }

    analysis_data = {}
    ticker = ticker if ticker.endswith('.NS') else f"{ticker}.NS"
    
    print(f"Fetching metrics for {ticker}")
    financial_metrics = ben_graham_metrics(ticker.split('.NS')[0], end_date, "annual", 10)
    
    if not financial_metrics:
        print(f"No metrics found for {ticker}")
        return
    
    print(f"Getting market cap for {ticker}")
    stock = yf.Ticker(ticker)
    price = get_closest_price(ticker, end_date)
    shares_outstanding = stock.info.get('sharesOutstanding', 0) / 1000000
    market_cap = price * shares_outstanding if price and shares_outstanding else None
    
    if not market_cap:
        latest_metrics = financial_metrics[0]
        market_cap = latest_metrics.net_income * 20 if latest_metrics.net_income else None
    
    if not market_cap:
        print(f"Could not determine market cap for {ticker}")
        return

    # Analysis Functions
    def analyze_earnings_stability(metrics):
        score = 0
        details = []
        eps_vals = [m.earnings_per_share for m in metrics if m.earnings_per_share is not None]
        if len(eps_vals) < 2:
            details.append("Not enough multi-year EPS data.")
        else:
            positive_eps_years = sum(1 for e in eps_vals if e > 0)
            total_eps_years = len(eps_vals)
            if positive_eps_years == total_eps_years:
                score += 3
                details.append("EPS was positive in all periods.")
            elif positive_eps_years >= (total_eps_years * 0.8):
                score += 2
                details.append("EPS was positive in most periods.")
            if eps_vals[-1] > eps_vals[0]:
                score += 1
                details.append("EPS grew over time.")
            else:
                details.append("EPS did not grow over time.")
        return {"score": score, "details": "; ".join(details)}

    def analyze_financial_strength(metrics):
        score = 0
        details = []
        latest = metrics[0]
        if latest.current_liabilities > 0:
            current_ratio = latest.current_assets / latest.current_liabilities
            if current_ratio >= 2.0:
                score += 2
                details.append(f"Current ratio = {current_ratio:.2f} (>=2.0).")
            elif current_ratio >= 1.5:
                score += 1
                details.append(f"Current ratio = {current_ratio:.2f} (>=1.5).")
        else:
            details.append("Current liabilities missing or zero.")
        if latest.total_assets > 0:
            debt_ratio = latest.total_liabilities / latest.total_assets
            if debt_ratio < 0.5:
                score += 2
                details.append(f"Debt ratio = {debt_ratio:.2f} (<0.5).")
            elif debt_ratio < 0.8:
                score += 1
                details.append(f"Debt ratio = {debt_ratio:.2f} (<0.8).")
        else:
            details.append("Total assets missing or zero.")
        div_periods = [m.dividends_per_share for m in metrics if m.dividends_per_share is not None]
        if div_periods and sum(1 for d in div_periods if d > 0) >= (len(div_periods) // 2 + 1):
            score += 1
            details.append("Dividends paid in majority of years.")
        elif not div_periods:
            details.append("No dividend data available.")
        return {"score": score, "details": "; ".join(details)}

    def analyze_valuation_graham(metrics, market_cap):
        score = 0
        details = []
        latest = metrics[0]
        
        # Debug data availability
        print(f"Debug Valuation - Ticker: {ticker}")
        print(f"Current Assets: {latest.current_assets}, Total Liabilities: {latest.total_liabilities}")
        print(f"EPS: {latest.earnings_per_share}, Book Value/Share: {latest.book_value_per_share}")
        print(f"Shares Outstanding: {latest.outstanding_shares}, Market Cap: {market_cap}")

        # NCAV Analysis
        if latest.current_assets is not None and latest.total_liabilities is not None:
            ncav = latest.current_assets - latest.total_liabilities
            if latest.outstanding_shares > 0:
                ncav_ps = ncav / latest.outstanding_shares
                price_ps = market_cap / latest.outstanding_shares
                details.append(f"NCAV = {ncav:,.2f}, NCAV/Share = {ncav_ps:.2f}, Price/Share = {price_ps:.2f}")
                if ncav > market_cap:
                    score += 4
                    details.append("NCAV > Market Cap (strong buy signal).")
                elif ncav_ps >= (price_ps * 0.67):
                    score += 2
                    details.append("NCAV >= 2/3 of Price (moderate value).")
                else:
                    details.append("NCAV below 2/3 of Price.")
            else:
                details.append("No shares outstanding for NCAV/Share calculation.")
        else:
            details.append("Missing current_assets or total_liabilities for NCAV.")

        # Graham Number Analysis
        if latest.earnings_per_share is not None and latest.book_value_per_share is not None:
            if latest.earnings_per_share > 0 and latest.book_value_per_share > 0:
                graham_number = math.sqrt(22.5 * latest.earnings_per_share * latest.book_value_per_share)
                current_price = market_cap / latest.outstanding_shares if latest.outstanding_shares > 0 else 0
                if current_price > 0:
                    margin_of_safety = (graham_number - current_price) / current_price
                    details.append(f"Graham Number = {graham_number:.2f}, Current Price = {current_price:.2f}, Margin of Safety = {margin_of_safety:.2%}")
                    if margin_of_safety > 0.5:
                        score += 3
                        details.append("Margin of Safety > 50% (undervalued).")
                    elif margin_of_safety > 0.2:
                        score += 1
                        details.append("Margin of Safety > 20% (some value).")
                    else:
                        details.append("Margin of Safety <= 20% (limited value).")
                else:
                    details.append("Current price invalid for margin of safety.")
            else:
                details.append("EPS or Book Value/Share not positive for Graham Number.")
        else:
            details.append("Missing EPS or Book Value/Share for Graham Number.")

        return {"score": score, "details": "; ".join(details)}

    # Perform analysis
    earnings_analysis = analyze_earnings_stability(financial_metrics)
    strength_analysis = analyze_financial_strength(financial_metrics)
    valuation_analysis = analyze_valuation_graham(financial_metrics, market_cap)

    total_score = earnings_analysis["score"] + strength_analysis["score"] + valuation_analysis["score"]
    max_score = 16
    signal = "bullish" if total_score >= 0.7 * max_score else "bearish" if total_score <= 0.3 * max_score else "neutral"

    analysis_data[ticker] = {
        "signal": signal,
        "score": total_score,
        "max_score": max_score,
        "earnings_analysis": earnings_analysis,
        "strength_analysis": strength_analysis,
        "valuation_analysis": valuation_analysis
    }

    # Display results
    print(f"\nBen Graham Analysis for {ticker}:")
    print(json.dumps(analysis_data, indent=2))

# Test the agent
test_ticker = "TCS"  # Using TCS as per your example
test_end_date = "2025-04-06"
test_ben_graham_agent(test_ticker, test_end_date)

Fetching metrics for TCS.NS


Getting market cap for TCS.NS
Debug Valuation - Ticker: TCS.NS
Current Assets: 72346.0, Total Liabilities: 48874.0
EPS: 120.39, Book Value/Share: 19.933169246461727
Shares Outstanding: 3618.089984, Market Cap: 11937525.7398805

Ben Graham Analysis for TCS.NS:
{
  "TCS.NS": {
    "signal": "neutral",
    "score": 7,
    "max_score": 16,
    "earnings_analysis": {
      "score": 3,
      "details": "EPS was positive in all periods.; EPS did not grow over time."
    },
    "strength_analysis": {
      "score": 4,
      "details": "Current ratio = 1.69 (>=1.5).; Debt ratio = 0.40 (<0.5).; Dividends paid in majority of years."
    },
    "valuation_analysis": {
      "score": 0,
      "details": "NCAV = 23,472.00, NCAV/Share = 6.49, Price/Share = 3299.40; NCAV below 2/3 of Price.; Graham Number = 232.37, Current Price = 3299.40, Margin of Safety = -92.96%; Margin of Safety <= 20% (limited value)."
    }
  }
}
